# A3 cross-session re-ID on Lee 2019 OpenBMI motor imagery (54 subjects)

Replicates the BCI IV-2a A3 cross-session result on a much larger cohort. Lee2019 has 54 subjects × 2 sessions on different days (binary left/right hand). Pipeline mirrors `experiments/05`: victim trained on session-1 task labels, re-ID probe trained on session-1 embeddings, tested on session-2 embeddings. If the IV-2a story (Riemann 91%, FBCSP 89%, EEGNet 78%) replicates here, A3 is no longer anecdotal at n=9.

Chance top-1 = 1/54 ≈ 1.85%. moabb downloads ~10 GB on first run; expect ~25 min just for the download.

Send back `results/20_a3_lee2019.json` and `figures/20_a3_lee2019.pdf`.

**Runtime → L4 GPU.** Expected wall ~50-60 min (incl. moabb download).

**Don't `Save a copy in GitHub` from Colab.**

## 1. Clone repo

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD

## 2. Confirm GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 3. Install deps

In [ ]:
import torch
tv = torch.__version__.split('+')[0]
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich

## 4. moabb cache warmup (~25 min, downloads ~10 GB to ~/mne_data)

In [ ]:
import moabb
from moabb.datasets import Lee2019_MI
ds = Lee2019_MI()
_ = ds.get_data(subjects=[1])  # triggers full Lee2019_MI download (one shared archive)
print('moabb Lee2019_MI cache ready')

## 5. Run experiment

Expected wall: ~50-60 min (incl. moabb download).

In [ ]:
!PYTHONUNBUFFERED=1 python -u -m experiments.20_a3_lee2019 --all

## 6. Emit run metadata + result JSON

In [ ]:
import json, os, subprocess, datetime, platform, sys
import torch
PROJECT_DIR = '/content/bci-identity-leakage'
if os.path.isdir(PROJECT_DIR): os.chdir(PROJECT_DIR)
EXPERIMENT = "experiments.20_a3_lee2019"
ARGS = ['--all']
RESULTS = 'results/20_a3_lee2019.json'
EXTRA_OUTPUTS = ['figures/20_a3_lee2019.pdf']
TAG = "a3_lee2019"
try:
    git_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_DIR).decode().strip()
except Exception as exc:
    print(f'  git unavailable ({exc}); using "unknown"'); git_sha = 'unknown'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
now_utc = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
run_id = now_utc.replace(':','').replace('-','').rstrip('Z') + f'_{TAG}_{git_sha[:7]}'
meta = {'run_id': run_id, 'experiment': EXPERIMENT, 'args': ARGS,
        'git_sha': git_sha, 'hardware': f'Colab {gpu}',
        'platform': platform.platform(), 'torch_version': torch.__version__,
        'completed_at_utc': now_utc, 'outputs': [RESULTS] + EXTRA_OUTPUTS}
os.makedirs(f'runs/{run_id}', exist_ok=True)
with open(f'runs/{run_id}/meta.json', 'w') as f: json.dump(meta, f, indent=2)
print('=== run metadata ==='); print(json.dumps(meta, indent=2)); print()
if not os.path.exists(RESULTS):
    sys.exit(f'!!! {RESULTS} missing — run cell did not finish. Re-run the run cell, then this cell.')
print(f'=== {RESULTS} ==='); print(open(RESULTS).read())


## 7. Download artifacts

In [ ]:
from google.colab import files
files.download('results/20_a3_lee2019.json')
files.download('figures/20_a3_lee2019.pdf')
files.download(f'runs/{run_id}/meta.json')
